# Training for Robustness — Full Experiment Runner

**Capstone:** Comparing Degradation Augmentation Strategies in CNN Image Classification

This notebook runs the full experimental matrix (50 runs) on Google Colab GPU,
saving results to Google Drive after each completed run.

## Setup
1. Mount Google Drive
2. Clone/upload the repo
3. Configure which experiments to run
4. Run all — results flush to Drive after each seed

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create project directory on Drive for persistent results
import os
DRIVE_DIR = '/content/drive/MyDrive/capstone_robustness'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'baseline'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'augmented'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'baseline', 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'augmented', 'checkpoints'), exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
!pip install torch torchvision scikit-learn --quiet

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Core Code
All code is self-contained below — no external imports needed.

In [ ]:
import io
import math
import os
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader
from PIL import Image, ImageFilter, ImageEnhance, ImageOps
from scipy.ndimage import convolve1d


# ═══════════════════════════════════════════════════════════════════════════
# DEGRADATION GRID
# ═══════════════════════════════════════════════════════════════════════════
DEG_GRID = {
    'gaussian_noise': [0.02, 0.05, 0.10],
    'gaussian_blur':  [3, 5, 7],
    'motion_blur':    [3, 5, 7],
    'jpeg':           [50, 30, 10],
    'contrast':       [0.8, 0.6, 0.4],
    'darkening':      [0.8, 0.6, 0.4],
}


# ═══════════════════════════════════════════════════════════════════════════
# DEGRADATION FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════
def apply_gaussian_noise(img_tensor, sigma):
    noise = torch.randn_like(img_tensor) * sigma
    return torch.clamp(img_tensor + noise, 0.0, 1.0)

def apply_gaussian_blur(img, kernel_size):
    return img.filter(ImageFilter.GaussianBlur(radius=kernel_size // 2))

def apply_motion_blur(img, kernel_size):
    img_np = np.array(img).astype(np.float64)
    kernel = np.ones(kernel_size) / kernel_size
    blurred = convolve1d(img_np, kernel, axis=1, mode='nearest')
    return Image.fromarray(np.clip(blurred, 0, 255).astype(np.uint8))

def apply_jpeg_compression(img, quality):
    buffer = io.BytesIO()
    img.save(buffer, format='JPEG', quality=quality)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

def apply_contrast(img, factor):
    return ImageEnhance.Contrast(img).enhance(factor)

def apply_darkening(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

_DEG_FNS = {
    'gaussian_noise': (apply_gaussian_noise, 'tensor'),
    'gaussian_blur':  (apply_gaussian_blur,  'pil'),
    'motion_blur':    (apply_motion_blur,    'pil'),
    'jpeg':           (apply_jpeg_compression, 'pil'),
    'contrast':       (apply_contrast,       'pil'),
    'darkening':      (apply_darkening,      'pil'),
}

def apply_degradation(img_tensor, deg_type, severity_value):
    fn, input_type = _DEG_FNS[deg_type]
    if input_type == 'tensor':
        return fn(img_tensor, severity_value)
    img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    pil_img = Image.fromarray(img_np)
    pil_img = fn(pil_img, severity_value)
    img_np = np.array(pil_img).astype(np.float32) / 255.0
    return torch.from_numpy(img_np).permute(2, 0, 1)


# ═══════════════════════════════════════════════════════════════════════════
# CURRICULUM SCHEDULER (linear, cosine, capped)
# ═══════════════════════════════════════════════════════════════════════════
def get_curriculum_severity(deg_type, epoch, total_epochs=200, schedule='linear', max_severity=None):
    levels = DEG_GRID[deg_type]
    ramp_end = total_epochs // 2
    if epoch <= 1:
        progress = 0.0
    elif epoch >= ramp_end:
        progress = 1.0
    else:
        progress = (epoch - 1) / (ramp_end - 1)

    # Apply schedule transform
    if schedule == 'cosine':
        progress = 0.5 * (1 - math.cos(math.pi * progress))

    mild = levels[0]
    if schedule == 'capped':
        hard = levels[1] if max_severity is None else max_severity
    else:
        hard = levels[-1] if max_severity is None else max_severity

    value = mild + progress * (hard - mild)
    if deg_type in ('gaussian_blur', 'motion_blur'):
        value = int(round(value))
        if value % 2 == 0:
            value += 1
    return value


# ═══════════════════════════════════════════════════════════════════════════
# AUGMIX OPERATIONS
# ═══════════════════════════════════════════════════════════════════════════
def _am_autocontrast(img, _): return ImageOps.autocontrast(img)
def _am_equalize(img, _): return ImageOps.equalize(img)
def _am_posterize(img, lv): return ImageOps.posterize(img, max(1, 4 - lv))
def _am_rotate(img, lv):
    deg = (lv / 3.0) * 30
    a = deg if np.random.random() > 0.5 else -deg
    return img.rotate(a, resample=Image.BILINEAR, fillcolor=(128,128,128))
def _am_solarize(img, lv): return ImageOps.solarize(img, 256 - int((lv/3)*128))
def _am_shear_x(img, lv):
    v = (lv/3)*0.3; v = v if np.random.random()>0.5 else -v
    return img.transform(img.size, Image.AFFINE, (1,v,0,0,1,0), resample=Image.BILINEAR, fillcolor=(128,128,128))
def _am_shear_y(img, lv):
    v = (lv/3)*0.3; v = v if np.random.random()>0.5 else -v
    return img.transform(img.size, Image.AFFINE, (1,0,0,v,1,0), resample=Image.BILINEAR, fillcolor=(128,128,128))
def _am_translate_x(img, lv):
    px = int((lv/3)*10); px = px if np.random.random()>0.5 else -px
    return img.transform(img.size, Image.AFFINE, (1,0,px,0,1,0), resample=Image.BILINEAR, fillcolor=(128,128,128))
def _am_translate_y(img, lv):
    px = int((lv/3)*10); px = px if np.random.random()>0.5 else -px
    return img.transform(img.size, Image.AFFINE, (1,0,0,0,1,px), resample=Image.BILINEAR, fillcolor=(128,128,128))
def _am_contrast(img, lv):
    f = 1.0+(lv/3)*0.9; f = f if np.random.random()>0.5 else 2.0-f
    return ImageEnhance.Contrast(img).enhance(max(0.1, f))
def _am_brightness(img, lv):
    f = 1.0+(lv/3)*0.9; f = f if np.random.random()>0.5 else 2.0-f
    return ImageEnhance.Brightness(img).enhance(max(0.1, f))
def _am_sharpness(img, lv):
    f = 1.0+(lv/3)*0.9; f = f if np.random.random()>0.5 else 2.0-f
    return ImageEnhance.Sharpness(img).enhance(max(0.1, f))

AUGMIX_OPS = [_am_autocontrast, _am_equalize, _am_posterize, _am_rotate, _am_solarize,
              _am_shear_x, _am_shear_y, _am_translate_x, _am_translate_y,
              _am_contrast, _am_brightness, _am_sharpness]

def augmix_single_image(pil_img, severity=3, width=3, depth=-1, alpha=1.0):
    img_np = np.array(pil_img).astype(np.float32) / 255.0
    ws = np.random.dirichlet([alpha] * width)
    m = np.random.beta(alpha, alpha)
    mix = np.zeros_like(img_np)
    for i in range(width):
        chain_img = pil_img.copy()
        d = depth if depth > 0 else np.random.randint(1, 4)
        for _ in range(d):
            op = np.random.choice(AUGMIX_OPS)
            chain_img = op(chain_img, severity)
        mix += ws[i] * (np.array(chain_img).astype(np.float32) / 255.0)
    result = m * img_np + (1 - m) * mix
    return Image.fromarray(np.clip(result * 255, 0, 255).astype(np.uint8))

def augmix_batch(images, severity=3):
    out = []
    for img_t in images:
        img_np = (img_t.permute(1,2,0).numpy()*255).astype(np.uint8)
        pil_img = Image.fromarray(img_np)
        aug_img = augmix_single_image(pil_img, severity=severity)
        aug_np = np.array(aug_img).astype(np.float32) / 255.0
        out.append(torch.from_numpy(aug_np).permute(2, 0, 1))
    return torch.stack(out)

def jsd_loss(logits_clean, logits_aug1, logits_aug2):
    p_clean = F.softmax(logits_clean, dim=1)
    p_aug1 = F.softmax(logits_aug1, dim=1)
    p_aug2 = F.softmax(logits_aug2, dim=1)
    p_mean = (p_clean + p_aug1 + p_aug2) / 3.0
    kl0 = F.kl_div(p_mean.log(), p_clean, reduction='batchmean')
    kl1 = F.kl_div(p_mean.log(), p_aug1, reduction='batchmean')
    kl2 = F.kl_div(p_mean.log(), p_aug2, reduction='batchmean')
    return (kl0 + kl1 + kl2) / 3.0


# ═══════════════════════════════════════════════════════════════════════════
# MODELS
# ═══════════════════════════════════════════════════════════════════════════
def resnet18_cifar10():
    model = models.resnet18(weights=None, num_classes=10)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

def mobilenetv2_cifar10():
    model = models.mobilenet_v2(weights=None, num_classes=10)
    first_conv = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        first_conv.in_channels, first_conv.out_channels,
        kernel_size=first_conv.kernel_size, stride=1,
        padding=first_conv.padding, bias=False,
    )
    return model

def get_model(name):
    if name == 'resnet18':
        return resnet18_cifar10()
    elif name == 'mobilenetv2':
        return mobilenetv2_cifar10()
    raise ValueError(f'Unknown model: {name}')


# ═══════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════
CIFAR10_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
CIFAR10_STD  = torch.tensor([0.2470, 0.2435, 0.2616]).view(3, 1, 1)
SEEDS = [0, 42, 123, 456, 789]
EPOCHS = 200
BATCH_SIZE = 128
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MILESTONES = [100, 150]
GAMMA = 0.1

print('All code loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_train_loader():
    transform = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
    ])
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform
    )
    return DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)

def get_test_loader():
    transform = T.ToTensor()
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform
    )
    return DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

def normalize_batch(images, device):
    mean = CIFAR10_MEAN.to(device)
    std = CIFAR10_STD.to(device)
    return (images - mean) / std

# Augmentation functions
def augment_single(images, deg_type):
    levels = DEG_GRID[deg_type]
    return torch.stack([apply_degradation(img, deg_type, random.choice(levels)) for img in images])

def augment_mixed(images):
    deg_types = list(DEG_GRID.keys())
    out = []
    for img in images:
        dt = random.choice(deg_types)
        out.append(apply_degradation(img, dt, random.choice(DEG_GRID[dt])))
    return torch.stack(out)

def augment_curriculum(images, deg_type, epoch):
    sev = get_curriculum_severity(deg_type, epoch, EPOCHS)
    return torch.stack([apply_degradation(img, deg_type, sev) for img in images])

def augment_curriculum_capped(images, deg_type, epoch):
    sev = get_curriculum_severity(deg_type, epoch, EPOCHS, schedule='capped')
    return torch.stack([apply_degradation(img, deg_type, sev) for img in images])

def augment_curriculum_cosine(images, deg_type, epoch):
    sev = get_curriculum_severity(deg_type, epoch, EPOCHS, schedule='cosine')
    return torch.stack([apply_degradation(img, deg_type, sev) for img in images])

def augment_curriculum_clean50(images, deg_type, epoch):
    sev = get_curriculum_severity(deg_type, epoch, EPOCHS)
    out = []
    for img in images:
        if random.random() < 0.5:
            out.append(apply_degradation(img, deg_type, sev))
        else:
            out.append(img)
    return torch.stack(out)

print('Helpers loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EVALUATION
# ═══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_accuracy(model, device, deg_type=None, severity_value=None):
    """Evaluate on clean or degraded test set. Returns accuracy %."""
    model.eval()
    loader = get_test_loader()
    correct = total = 0
    for images, labels in loader:
        if deg_type is not None:
            images = torch.stack([apply_degradation(img, deg_type, severity_value) for img in images])
        images = normalize_batch(images.to(device), device)
        labels = labels.to(device)
        _, predicted = model(images).max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return 100.0 * correct / total

def full_eval(model, device, model_name, strategy, seed, deg_trained='none'):
    """Full robustness evaluation: clean + 18 conditions."""
    results = []
    acc = evaluate_accuracy(model, device)
    results.append({'model': model_name, 'strategy': strategy, 'deg_type_trained': deg_trained,
                    'seed': seed, 'eval_degradation': 'clean', 'eval_severity': 0, 'accuracy': acc})
    print(f'  Clean: {acc:.2f}%')
    
    for deg_name, levels in DEG_GRID.items():
        for sev_idx, sev_val in enumerate(levels, 1):
            acc = evaluate_accuracy(model, device, deg_name, sev_val)
            results.append({'model': model_name, 'strategy': strategy, 'deg_type_trained': deg_trained,
                            'seed': seed, 'eval_degradation': deg_name, 'eval_severity': sev_idx, 'accuracy': acc})
            print(f'  {deg_name} sev{sev_idx}: {acc:.2f}%')
    return results

print('Evaluation loaded.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# TRAINING + SAVE TO DRIVE (with per-epoch accuracy logging)
# ═══════════════════════════════════════════════════════════════════════════

STRATEGIES_WITH_DEG = ('single', 'curriculum', 'curriculum_capped', 'curriculum_cosine', 'curriculum_clean50')
LOG_EVERY = 20  # evaluate clean accuracy every N epochs

def run_experiment(strategy, model_name, seed, deg_type='none', drive_dir=DRIVE_DIR):
    """Train one model, evaluate, save results + checkpoint + epoch log to Drive."""
    
    # Build file paths
    subdir = 'baseline' if strategy == 'baseline' else 'augmented'
    if strategy in STRATEGIES_WITH_DEG:
        csv_name = f'{model_name}_{strategy}_{deg_type}_robustness.csv'
        ckpt_name = f'{model_name}_{strategy}_{deg_type}_seed{seed}.pt'
        log_name = f'{model_name}_{strategy}_{deg_type}_seed{seed}_epochlog.csv'
    else:
        csv_name = f'{model_name}_{strategy}_robustness.csv'
        ckpt_name = f'{model_name}_{strategy}_seed{seed}.pt'
        log_name = f'{model_name}_{strategy}_seed{seed}_epochlog.csv'
    
    csv_path = os.path.join(drive_dir, subdir, csv_name)
    ckpt_path = os.path.join(drive_dir, subdir, 'checkpoints', ckpt_name)
    log_path = os.path.join(drive_dir, subdir, 'epoch_logs', log_name)
    
    # Skip if this seed already exists in CSV (fully completed)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        if seed in existing['seed'].values:
            print(f'SKIP: {strategy}/{model_name}/seed{seed}/{deg_type} already done')
            return None
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'\n{"="*60}')
    print(f'Strategy: {strategy} | Model: {model_name} | Seed: {seed} | Deg: {deg_type}')
    print(f'Device: {device}')
    print(f'{"="*60}')
    
    model = get_model(model_name).to(device)
    
    # If checkpoint exists but CSV doesn't, skip training and just re-evaluate
    if os.path.exists(ckpt_path):
        print(f'RECOVER: Loading checkpoint (training was done, eval crashed)')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        epoch_log = []  # no log available for recovered runs
    else:
        # Full training
        set_seed(seed)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)
        loader = get_train_loader()
        epoch_log = []
        
        start = time.time()
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            total_samples = 0
            
            for images, labels in loader:
                # Apply augmentation
                if strategy == 'single':
                    images = augment_single(images, deg_type)
                elif strategy == 'mixed':
                    images = augment_mixed(images)
                elif strategy == 'curriculum':
                    images = augment_curriculum(images, deg_type, epoch)
                elif strategy == 'curriculum_capped':
                    images = augment_curriculum_capped(images, deg_type, epoch)
                elif strategy == 'curriculum_cosine':
                    images = augment_curriculum_cosine(images, deg_type, epoch)
                elif strategy == 'curriculum_clean50':
                    images = augment_curriculum_clean50(images, deg_type, epoch)
                elif strategy in ('augmix', 'augmix_nojsd'):
                    pass  # handled below
                # baseline: no augmentation
                
                labels = labels.to(device)
                
                if strategy == 'augmix':
                    images_clean = normalize_batch(images.to(device), device)
                    images_aug1 = normalize_batch(augmix_batch(images).to(device), device)
                    images_aug2 = normalize_batch(augmix_batch(images).to(device), device)
                    optimizer.zero_grad()
                    logits_clean = model(images_clean)
                    logits_aug1 = model(images_aug1)
                    logits_aug2 = model(images_aug2)
                    ce_loss = criterion(logits_clean, labels)
                    js_loss = jsd_loss(logits_clean, logits_aug1, logits_aug2)
                    loss = ce_loss + 12.0 * js_loss
                    loss.backward()
                    optimizer.step()
                elif strategy == 'augmix_nojsd':
                    images = normalize_batch(augmix_batch(images).to(device), device)
                    optimizer.zero_grad()
                    loss = criterion(model(images), labels)
                    loss.backward()
                    optimizer.step()
                else:
                    images = normalize_batch(images.to(device), device)
                    optimizer.zero_grad()
                    loss = criterion(model(images), labels)
                    loss.backward()
                    optimizer.step()
                
                total_loss += loss.item() * labels.size(0)
                total_samples += labels.size(0)
            
            scheduler.step()
            avg_loss = total_loss / total_samples
            
            # Per-epoch clean accuracy logging
            if epoch % LOG_EVERY == 0 or epoch == 1 or epoch == EPOCHS:
                clean_acc = evaluate_accuracy(model, device)
                elapsed = time.time() - start
                epoch_log.append({'epoch': epoch, 'loss': avg_loss, 'clean_acc': clean_acc})
                print(f'  Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.4f} | Clean: {clean_acc:.2f}% | Time: {elapsed:.0f}s')
        
        train_time = time.time() - start
        print(f'Training complete in {train_time:.0f}s ({train_time/60:.1f}min)')
        
        # Save checkpoint to Drive
        os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
        torch.save(model.state_dict(), ckpt_path)
        print(f'Checkpoint: {ckpt_path}')
        
        # Save epoch log
        if epoch_log:
            os.makedirs(os.path.dirname(log_path), exist_ok=True)
            pd.DataFrame(epoch_log).to_csv(log_path, index=False)
            print(f'Epoch log: {log_path}')
    
    # Evaluate
    print('Evaluating robustness...')
    results = full_eval(model, device, model_name, strategy, seed, deg_type)
    
    # Save/append CSV to Drive
    df = pd.DataFrame(results)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(csv_path, index=False)
    print(f'Results saved: {csv_path}')
    
    return df

## Configuration

Edit the cell below to control which experiments to run.  
Set `RUN_ALL = True` to run the full matrix, or customize the lists.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell
# ═══════════════════════════════════════════════════════════════════════════
RUN_ORIGINAL = False    # Original 60 experiments (already done)
RUN_EXTENSIONS = True   # 50 new experiments (augmix, capped, clean50, cosine)
RUN_AUGMIX_NOJSD = True # 10 more (augmix without JSD loss — ablation)
RUN_EPOCH_LOGGING = True # Re-run 1 seed of key original strategies to get epoch curves

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BUILD EXPERIMENT LIST
# ═══════════════════════════════════════════════════════════════════════════
experiments = []

MODELS_LIST = ['resnet18', 'mobilenetv2']
SINGLE_DEGS = ['gaussian_blur', 'gaussian_noise']

if RUN_ORIGINAL:
    # Baseline: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('baseline', m, s, 'none'))
    # Mixed: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('mixed', m, s, 'none'))
    # Single: 2 × 2 × 5 = 20
    for d in SINGLE_DEGS:
        for m in MODELS_LIST:
            for s in SEEDS:
                experiments.append(('single', m, s, d))
    # Curriculum: 2 × 2 × 5 = 20
    for d in SINGLE_DEGS:
        for m in MODELS_LIST:
            for s in SEEDS:
                experiments.append(('curriculum', m, s, d))

if RUN_EXTENSIONS:
    # AugMix: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('augmix', m, s, 'none'))
    # Capped curriculum (blur only): 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('curriculum_capped', m, s, 'gaussian_blur'))
    # Curriculum with 50% clean (blur only): 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('curriculum_clean50', m, s, 'gaussian_blur'))
    # Cosine curriculum: 2 × 2 × 5 = 20
    for d in SINGLE_DEGS:
        for m in MODELS_LIST:
            for s in SEEDS:
                experiments.append(('curriculum_cosine', m, s, d))

if RUN_AUGMIX_NOJSD:
    # AugMix without JSD: 2 × 5 = 10
    for m in MODELS_LIST:
        for s in SEEDS:
            experiments.append(('augmix_nojsd', m, s, 'none'))

if RUN_EPOCH_LOGGING:
    # Re-run 1 seed of key original strategies just for epoch logging
    # These will get new epoch_log CSVs but the robustness CSV will be skipped (already exists)
    # IMPORTANT: delete the existing robustness CSV entry for seed 42 first,
    # or use a different seed that hasn't been run yet
    # Actually we use seed 99 which is NOT in the original seeds, so it won't skip
    LOGGING_SEED = 99
    for m in MODELS_LIST:
        experiments.append(('baseline', m, LOGGING_SEED, 'none'))
        experiments.append(('mixed', m, LOGGING_SEED, 'none'))
        experiments.append(('curriculum', m, LOGGING_SEED, 'gaussian_blur'))
        experiments.append(('curriculum', m, LOGGING_SEED, 'gaussian_noise'))
        experiments.append(('single', m, LOGGING_SEED, 'gaussian_noise'))

n_ext = sum([50 * RUN_EXTENSIONS, 10 * RUN_AUGMIX_NOJSD, 10 * RUN_EPOCH_LOGGING])
print(f'Total experiments: {len(experiments)}')
print(f'  Extensions: {50 if RUN_EXTENSIONS else 0}')
print(f'  AugMix no-JSD: {10 if RUN_AUGMIX_NOJSD else 0}')
print(f'  Epoch logging re-runs: {10 if RUN_EPOCH_LOGGING else 0}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN ALL EXPERIMENTS
# ═══════════════════════════════════════════════════════════════════════════
# Already-completed runs are automatically skipped.
# If Colab disconnects, just re-run this cell to resume.

for i, (strategy, model_name, seed, deg_type) in enumerate(experiments, 1):
    print(f'\n[{i}/{len(experiments)}]')
    run_experiment(strategy, model_name, seed, deg_type)

print('\n\nALL EXPERIMENTS COMPLETE!')

## Verify Results
Check what's been completed so far.

In [ ]:
import glob

csvs = glob.glob(os.path.join(DRIVE_DIR, '**', '*_robustness.csv'), recursive=True)
print(f'Found {len(csvs)} result files:\n')
total_seeds = 0
for f in sorted(csvs):
    df = pd.read_csv(f)
    n_seeds = df['seed'].nunique()
    total_seeds += n_seeds
    print(f'  {os.path.basename(f)}: {n_seeds} seeds')
print(f'\nTotal completed runs: {total_seeds}/100')
print(f'  (50 original + 50 extensions)')

## Grad-CAM Visualization
Generate heatmaps showing what each training strategy focuses on.
Run this after all experiments are complete.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GRAD-CAM
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.cm as cm_color

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        self._fwd = target_layer.register_forward_hook(self._save_act)
        self._bwd = target_layer.register_full_backward_hook(self._save_grad)
    
    def _save_act(self, mod, inp, out): self.activations = out.detach()
    def _save_grad(self, mod, gi, go): self.gradients = go[0].detach()
    
    def __call__(self, x, class_idx=None):
        self.model.eval()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        self.model.zero_grad()
        out[0, class_idx].backward()
        w = self.gradients.mean(dim=(2,3), keepdim=True)
        heatmap = F.relu((w * self.activations).sum(1, keepdim=True))
        heatmap = heatmap.squeeze().cpu().numpy()
        if heatmap.max() > 0: heatmap /= heatmap.max()
        heatmap = F.interpolate(
            torch.from_numpy(heatmap).unsqueeze(0).unsqueeze(0).float(),
            size=(x.shape[2], x.shape[3]), mode='bilinear', align_corners=False
        ).squeeze().numpy()
        return heatmap
    
    def remove(self):
        self._fwd.remove()
        self._bwd.remove()

def get_target_layer(model, name):
    if name == 'resnet18': return model.layer4[-1]
    elif name == 'mobilenetv2': return model.features[-1]

def overlay(img_np, heatmap, alpha=0.4):
    colormap = cm_color.jet(heatmap)[:,:,:3]
    return np.clip((1-alpha)*img_np + alpha*colormap, 0, 1)

# Pick representative test images (one per class)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=T.ToTensor())
class_imgs = {}
for img, lbl in testset:
    if lbl not in class_imgs: class_imgs[lbl] = img
    if len(class_imgs) >= 8: break
images = [class_imgs[k] for k in sorted(class_imgs)[:8]]
CLASSES = ['airplane','auto','bird','cat','deer','dog','frog','horse','ship','truck']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for model_name in ['resnet18', 'mobilenetv2']:
    # Define which checkpoints to compare
    ckpts = {}
    base_ckpt = os.path.join(DRIVE_DIR, 'baseline', 'checkpoints', f'{model_name}_baseline_seed0.pt')
    mixed_ckpt = os.path.join(DRIVE_DIR, 'augmented', 'checkpoints', f'{model_name}_mixed_seed0.pt')
    curblur_ckpt = os.path.join(DRIVE_DIR, 'augmented', 'checkpoints', f'{model_name}_curriculum_gaussian_blur_seed0.pt')
    augmix_ckpt = os.path.join(DRIVE_DIR, 'augmented', 'checkpoints', f'{model_name}_augmix_seed0.pt')
    
    if os.path.exists(base_ckpt): ckpts['baseline'] = base_ckpt
    if os.path.exists(mixed_ckpt): ckpts['mixed'] = mixed_ckpt
    if os.path.exists(curblur_ckpt): ckpts['curric/blur'] = curblur_ckpt
    if os.path.exists(augmix_ckpt): ckpts['augmix'] = augmix_ckpt
    
    if not ckpts:
        print(f'No checkpoints found for {model_name}, skipping GradCAM')
        continue
    
    conditions = [('clean', None, None), ('blur s3', 'gaussian_blur', 7), ('noise s3', 'gaussian_noise', 0.10)]
    n_cols = len(ckpts) * len(conditions)
    
    fig, axes = plt.subplots(8, n_cols, figsize=(2.2*n_cols, 18))
    if n_cols == 1:
        axes = axes.reshape(-1, 1)
    
    for si, (slabel, spath) in enumerate(ckpts.items()):
        model = get_model(model_name).to(device)
        model.load_state_dict(torch.load(spath, map_location=device, weights_only=True))
        gcam = GradCAM(model, get_target_layer(model, model_name))
        
        for ci, (cname, dtyp, sval) in enumerate(conditions):
            col = si * len(conditions) + ci
            for row, img_t in enumerate(images):
                vis = apply_degradation(img_t, dtyp, sval) if dtyp else img_t.clone()
                norm = ((vis - CIFAR10_MEAN.squeeze().view(3,1,1)) / CIFAR10_STD.squeeze().view(3,1,1)).unsqueeze(0).to(device)
                hm = gcam(norm)
                ov = overlay(vis.permute(1,2,0).numpy(), hm)
                axes[row, col].imshow(ov)
                axes[row, col].axis('off')
                if row == 0: axes[row, col].set_title(f'{slabel}\n{cname}', fontsize=7)
                if col == 0:
                    lbl = sorted(class_imgs)[:8][row]
                    axes[row, col].set_ylabel(CLASSES[lbl], fontsize=7, rotation=0, labelpad=35)
        gcam.remove()
    
    plt.suptitle(f'Grad-CAM: {model_name}', fontsize=13, y=1.01)
    plt.tight_layout()
    out_path = os.path.join(DRIVE_DIR, f'gradcam_{model_name}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

## Per-Epoch Accuracy Curves
Plot clean accuracy over training epochs for each strategy. Shows exactly when curriculum/blur collapses.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PER-EPOCH ACCURACY CURVES
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

log_dir_base = os.path.join(DRIVE_DIR, 'baseline', 'epoch_logs')
log_dir_aug = os.path.join(DRIVE_DIR, 'augmented', 'epoch_logs')

for model_name in ['resnet18', 'mobilenetv2']:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Strategies to plot with their log file patterns
    curves = [
        ('baseline', os.path.join(log_dir_base, f'{model_name}_baseline_seed99_epochlog.csv'), 'black', '-'),
        ('mixed', os.path.join(log_dir_aug, f'{model_name}_mixed_seed99_epochlog.csv'), 'green', '-'),
        ('single/noise', os.path.join(log_dir_aug, f'{model_name}_single_gaussian_noise_seed99_epochlog.csv'), 'blue', '--'),
        ('curric/noise', os.path.join(log_dir_aug, f'{model_name}_curriculum_gaussian_noise_seed99_epochlog.csv'), 'orange', '--'),
        ('curric/blur', os.path.join(log_dir_aug, f'{model_name}_curriculum_gaussian_blur_seed99_epochlog.csv'), 'red', '-'),
    ]
    
    # Also look for extension epoch logs (these use seed 0 from the extension runs)
    ext_curves = [
        ('curric_capped/blur', os.path.join(log_dir_aug, f'{model_name}_curriculum_capped_gaussian_blur_seed0_epochlog.csv'), 'purple', '-.'),
        ('curric_clean50/blur', os.path.join(log_dir_aug, f'{model_name}_curriculum_clean50_gaussian_blur_seed0_epochlog.csv'), 'brown', '-.'),
        ('curric_cosine/blur', os.path.join(log_dir_aug, f'{model_name}_curriculum_cosine_gaussian_blur_seed0_epochlog.csv'), 'magenta', '-.'),
        ('augmix', os.path.join(log_dir_aug, f'{model_name}_augmix_seed0_epochlog.csv'), 'teal', '-'),
    ]
    curves.extend(ext_curves)
    
    found_any = False
    for label, path, color, style in curves:
        if os.path.exists(path):
            log_df = pd.read_csv(path)
            ax.plot(log_df['epoch'], log_df['clean_acc'], label=label, color=color, linestyle=style, linewidth=2)
            found_any = True
    
    if not found_any:
        print(f'No epoch logs found for {model_name}. Run the epoch logging experiments first.')
        plt.close()
        continue
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Clean Test Accuracy (%)')
    ax.set_title(f'{model_name} — Clean Accuracy During Training')
    ax.axvline(x=100, color='grey', linestyle=':', alpha=0.5, label='LR drop / ramp end')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_path = os.path.join(DRIVE_DIR, f'epoch_curves_{model_name}.png')
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved: {out_path}')

## Confusion Matrices
Compare what the baseline, mixed, and curriculum/blur models get wrong.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFUSION MATRICES
# ═══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

CLASSES = ['airplane','auto','bird','cat','deer','dog','frog','horse','ship','truck']

@torch.no_grad()
def get_predictions(model, device, deg_type=None, sev_val=None):
    """Get all predictions and true labels on test set."""
    model.eval()
    loader = get_test_loader()
    all_preds, all_labels = [], []
    for images, labels in loader:
        if deg_type is not None:
            images = torch.stack([apply_degradation(img, deg_type, sev_val) for img in images])
        images = normalize_batch(images.to(device), device)
        _, predicted = model(images).max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for model_name in ['resnet18', 'mobilenetv2']:
    # Load checkpoints
    ckpts = {}
    base = os.path.join(DRIVE_DIR, 'baseline', 'checkpoints', f'{model_name}_baseline_seed0.pt')
    mixed = os.path.join(DRIVE_DIR, 'augmented', 'checkpoints', f'{model_name}_mixed_seed0.pt')
    curblur = os.path.join(DRIVE_DIR, 'augmented', 'checkpoints', f'{model_name}_curriculum_gaussian_blur_seed0.pt')
    
    if os.path.exists(base): ckpts['baseline'] = base
    if os.path.exists(mixed): ckpts['mixed'] = mixed
    if os.path.exists(curblur): ckpts['curric/blur'] = curblur
    
    if not ckpts:
        print(f'No checkpoints for {model_name}')
        continue
    
    # Two conditions: clean and gaussian_blur severity 3
    conditions = [('Clean', None, None), ('Gauss Blur s3', 'gaussian_blur', 7)]
    
    n_strats = len(ckpts)
    n_conds = len(conditions)
    fig, axes = plt.subplots(n_conds, n_strats, figsize=(5*n_strats, 5*n_conds))
    if n_strats == 1: axes = axes.reshape(-1, 1)
    if n_conds == 1: axes = axes.reshape(1, -1)
    
    for si, (slabel, spath) in enumerate(ckpts.items()):
        model = get_model(model_name).to(device)
        model.load_state_dict(torch.load(spath, map_location=device, weights_only=True))
        
        for ci, (cname, dtyp, sval) in enumerate(conditions):
            labels, preds = get_predictions(model, device, dtyp, sval)
            acc = 100.0 * (labels == preds).mean()
            cm = confusion_matrix(labels, preds, normalize='true')
            
            disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
            disp.plot(ax=axes[ci, si], values_format='.2f', cmap='Blues', colorbar=False)
            axes[ci, si].set_title(f'{slabel} | {cname}\n({acc:.1f}%)', fontsize=10)
            axes[ci, si].tick_params(labelsize=7)
            # Rotate x labels
            axes[ci, si].set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=7)
            axes[ci, si].set_yticklabels(CLASSES, fontsize=7)
    
    plt.suptitle(f'Confusion Matrices: {model_name}', fontsize=14)
    plt.tight_layout()
    out_path = os.path.join(DRIVE_DIR, f'confusion_{model_name}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')